# Prompt 21: Schemas and Data Types
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## Why Schemas Matter

Every Spark DataFrame has a **schema** — a description of column names and their data types. Schemas control:
- How data is read from files (type interpretation)
- How operations behave (`cast`, arithmetic, comparisons)
- Whether null values are allowed
- Memory layout and serialization

---

## StructType and StructField

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField('name',   StringType(),  nullable=True),
    StructField('age',    IntegerType(), nullable=False),
    StructField('salary', DoubleType(),  nullable=True),
])
```

- **`StructType`** — the schema container; a list of `StructField` objects
- **`StructField(name, dataType, nullable=True)`** — defines one column
  - `name` — column name (string)
  - `dataType` — any Spark type (see table below)
  - `nullable` — whether the column can contain `NULL` (default `True`)

---

## Spark Data Types — Complete Reference

### Primitive Types

| Type | Import | Python equivalent | Notes |
|------|--------|-------------------|-------|
| `StringType()` | `from pyspark.sql.types import StringType` | `str` | Variable-length text |
| `IntegerType()` | — | `int` | 32-bit signed integer (−2.1B to +2.1B) |
| `LongType()` | — | `int` | 64-bit signed integer; default for `count()` result |
| `ShortType()` | — | `int` | 16-bit signed integer |
| `ByteType()` | — | `int` | 8-bit signed integer |
| `FloatType()` | — | `float` | 32-bit IEEE 754 float; less precise |
| `DoubleType()` | — | `float` | 64-bit IEEE 754 double; default for float literals |
| `DecimalType(p,s)` | — | `Decimal` | Exact decimal; `p`=precision, `s`=scale; avoids float rounding |
| `BooleanType()` | — | `bool` | `True` / `False` |
| `DateType()` | — | `datetime.date` | Date without time component |
| `TimestampType()` | — | `datetime.datetime` | Datetime with microsecond precision |
| `TimestampNTZType()` | — | `datetime.datetime` | Timestamp without timezone (Spark 3.4+) |
| `BinaryType()` | — | `bytes` / `bytearray` | Raw binary data (e.g., images) |
| `NullType()` | — | `None` | Column that only ever contains `null` |

### Complex Types

| Type | Syntax | Description |
|------|--------|-------------|
| `ArrayType` | `ArrayType(StringType())` | Ordered list of elements of a single type |
| `MapType` | `MapType(StringType(), IntegerType())` | Key-value pairs; all keys same type, all values same type |
| `StructType` | `StructType([StructField(...)])` | Nested record / struct — embedded schema |

---

## printSchema() — Reading the Output

```
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- scores: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)
```

Key fields in output:
- **`nullable = true/false`** — whether a null is allowed in that column
- **`containsNull`** — for ArrayType: whether array elements can be null
- **`valueContainsNull`** — for MapType: whether values can be null
- Nested structs are indented with `|--` prefix

---

## Accessing Nested and Complex Fields

```python
# StructType (nested) — dot notation
df.select('address.city')         # dot string — column becomes 'city'
df.select(col('address.city'))    # col() with dot notation
df['address']['city']             # chained bracket access

# ArrayType — index access
df.select(df['tags'][0])          # first element of array
df.select(col('tags')[1])         # second element

# MapType — key access
df.select(df['scores']['math'])   # value for key 'math'

# explode — one row per array element (or map key-value pair)
from pyspark.sql.functions import explode
df.select('id', explode('tags').alias('tag'))   # N rows per original row
```

In [ ]:
# Cell 1: Setup and primitive type schemas
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, FloatType, DoubleType,
    DecimalType, BooleanType, DateType, TimestampType,
    BinaryType, NullType, ShortType, ByteType
)
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date, to_timestamp, lit
from datetime import date, datetime
from decimal import Decimal

spark = SparkSession.builder \
    .appName('SchemasDataTypes') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# -- Define a schema with all primitive types --
primitive_schema = StructType([
    StructField('id',           IntegerType(),       nullable=False),
    StructField('name',         StringType(),        nullable=True),
    StructField('age_short',    ShortType(),         nullable=True),
    StructField('flag',         BooleanType(),       nullable=True),
    StructField('salary_float', FloatType(),         nullable=True),
    StructField('salary_dbl',   DoubleType(),        nullable=True),
    StructField('salary_dec',   DecimalType(10, 2),  nullable=True),
    StructField('hire_date',    DateType(),          nullable=True),
    StructField('last_login',   TimestampType(),     nullable=True),
])

rows = [
    (1, 'Alice', 30,  True,  75000.50, 75000.50, Decimal('75000.50'),
     date(2020, 1, 15), datetime(2024, 3, 10, 9, 0, 0)),
    (2, 'Bob',   45,  False, 92000.00, 92000.00, Decimal('92000.00'),
     date(2018, 6, 1),  datetime(2024, 3, 11, 14, 30, 0)),
    (3, None,    None, None, None,     None,      None,
     None,             None),
]

df_prim = spark.createDataFrame(rows, schema=primitive_schema)

print('=== printSchema() — primitive types ===')
df_prim.printSchema()

print('=== show() ===')
df_prim.show(truncate=False)

# Confirm declared types match
print('=== dtypes (list of (name, type_string) tuples) ===')
for name, dtype in df_prim.dtypes:
    print(f'  {name:15s}: {dtype}')

In [ ]:
# Cell 2: Complex types — ArrayType, MapType, nested StructType
from pyspark.sql.types import ArrayType, MapType
from pyspark.sql.functions import explode, explode_outer, map_keys, map_values

complex_schema = StructType([
    StructField('id',   IntegerType(), nullable=False),
    StructField('name', StringType(),  nullable=True),
    # Nested StructType — address is an embedded record
    StructField('address', StructType([
        StructField('street', StringType(), nullable=True),
        StructField('city',   StringType(), nullable=True),
        StructField('zip',    StringType(), nullable=True),
    ]), nullable=True),
    # ArrayType — list of tags
    StructField('tags', ArrayType(StringType(), containsNull=True), nullable=True),
    # MapType — subject -> score
    StructField('scores', MapType(StringType(), IntegerType(), valueContainsNull=True),
                nullable=True),
])

complex_rows = [
    (1, 'Alice',
     ('123 Main St', 'Toronto', 'M5V 1A1'),
     ['python', 'spark', 'sql'],
     {'math': 95, 'english': 88}),
    (2, 'Bob',
     ('456 Oak Ave', 'Vancouver', 'V6B 2P5'),
     ['java', 'scala'],
     {'math': 72, 'science': 85}),
    (3, 'Carol',
     None,      # null struct
     None,      # null array
     None),     # null map
]

df_complex = spark.createDataFrame(complex_rows, schema=complex_schema)

print('=== printSchema() — complex types ===')
df_complex.printSchema()

print('=== Accessing nested StructType field: address.city ===')
df_complex.select('id', 'address.city', 'address.zip').show()

print('=== col() dot notation for nested field ===')
df_complex.select('id', col('address.street')).show()

print('=== Accessing ArrayType elements by index ===')
df_complex.select(
    'id',
    col('tags')[0].alias('first_tag'),
    col('tags')[1].alias('second_tag')
).show()

print('=== explode() — one row per array element ===')
df_complex.select('id', explode('tags').alias('tag')).show()

print('=== explode_outer() — keeps rows even if array is null/empty ===')
df_complex.select('id', explode_outer('tags').alias('tag')).show()

print('=== Accessing MapType by key ===')
df_complex.select(
    'id',
    col('scores')['math'].alias('math_score')
).show()

print('=== map_keys() and map_values() ===')
df_complex.select('id', map_keys('scores').alias('subjects'),
                  map_values('scores').alias('values')).show()

print('=== explode() on MapType — one row per key-value pair ===')
df_complex.select('id', explode('scores').alias('subject', 'score')).show()

In [ ]:
# Cell 3: Schema inference vs explicit schema — inferSchema limitations
import tempfile, os
BASE = tempfile.mkdtemp().replace('\\', '/')

# Write a CSV with ambiguous types
csv_data = '''id,age,salary,hire_date,active,phone
1,30,75000.50,2020-01-15,true,416-555-0100
2,45,92000.00,2018-06-01,false,604-555-0200
3,,50000.00,2022-03-20,true,'''

csv_path = f'{BASE}/employees.csv'
with open(csv_path, 'w') as f:
    f.write(csv_data)

print('=== Without inferSchema — all columns read as STRING (default) ===')
df_no_infer = spark.read \
    .option('header', True) \
    .csv(csv_path)
df_no_infer.printSchema()
df_no_infer.show()
# All types are StringType — age, salary, hire_date are strings

print('=== With inferSchema=True — Spark samples file to guess types ===')
df_infer = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(csv_path)
df_infer.printSchema()
df_infer.show()
# Problems: phone='416-555-0100' may be read as string (good)
#           hire_date may be read as string (does not auto-parse dates)
#           active='true' may be read as boolean

print('=== Inference limitation: dates need explicit schema or cast ===')
print(f'hire_date type from inferSchema: '
      f'{dict(df_infer.dtypes)["hire_date"]}')
# Often reads as string; use explicit schema for reliability

print('=== Explicit schema — guaranteed types, faster read (no sampling pass) ===')
explicit_schema = StructType([
    StructField('id',        IntegerType(), nullable=True),
    StructField('age',       IntegerType(), nullable=True),
    StructField('salary',    DoubleType(),  nullable=True),
    StructField('hire_date', DateType(),    nullable=True),
    StructField('active',    BooleanType(), nullable=True),
    StructField('phone',     StringType(),  nullable=True),
])

df_explicit = spark.read \
    .option('header', True) \
    .option('dateFormat', 'yyyy-MM-dd') \
    .schema(explicit_schema) \
    .csv(csv_path)
df_explicit.printSchema()
df_explicit.show()
# hire_date is now DateType; age column with empty string → null (not error)

print('=== Trade-off summary ===')
print('inferSchema=True  : convenient, slower (scans file), may be wrong for edge cases')
print('Explicit schema   : reliable, faster read, required for dates/decimals/booleans')

In [ ]:
# Cell 4: cast() — changing column types after read

# Start with all-string DataFrame (simulates no-inferSchema read)
raw_data = [
    ('1', '30',   '75000.50', '2020-01-15', 'true'),
    ('2', '45',   '92000.00', '2018-06-01', 'false'),
    ('3', 'NONE', '50000.00', '2022-03-20', 'true'),   # 'NONE' is not a valid int
    ('4', None,   None,       None,         None),
]
df_raw = spark.createDataFrame(raw_data,
             ['id_str', 'age_str', 'salary_str', 'date_str', 'active_str'])
df_raw.printSchema()
df_raw.show()

print('=== cast() — basic type conversion ===')
df_cast = df_raw.select(
    col('id_str').cast(IntegerType()).alias('id'),
    col('age_str').cast(IntegerType()).alias('age'),   # 'NONE' → null (no error!)
    col('salary_str').cast(DoubleType()).alias('salary'),
    col('date_str').cast(DateType()).alias('hire_date'),
    col('active_str').cast(BooleanType()).alias('active'),
)
df_cast.printSchema()
df_cast.show()
# IMPORTANT: invalid cast (e.g., 'NONE' → int) returns NULL, not an exception

print('=== cast() using string type names (shorthand) ===')
df_shorthand = df_raw.select(
    col('id_str').cast('int').alias('id'),
    col('salary_str').cast('double').alias('salary'),
    col('date_str').cast('date').alias('hire_date'),
    col('active_str').cast('boolean').alias('active'),
)
df_shorthand.printSchema()
df_shorthand.show()

print('=== withColumn + cast — update a column in place ===')
df_updated = df_raw \
    .withColumn('age_str', col('age_str').cast('int')) \
    .withColumnRenamed('age_str', 'age')
df_updated.printSchema()

print('=== to_date() / to_timestamp() — explicit date parsing (preferred over cast for date strings) ===')
df_dates = df_raw.select(
    col('id_str').cast('int').alias('id'),
    to_date(col('date_str'), 'yyyy-MM-dd').alias('hire_date'),
    to_timestamp(col('date_str'), 'yyyy-MM-dd').alias('hire_ts'),
)
df_dates.printSchema()
df_dates.show(truncate=False)

print('=== DecimalType — exact precision for financial data ===')
df_dec = df_raw.select(
    col('id_str').cast('int').alias('id'),
    col('salary_str').cast(DecimalType(10, 2)).alias('salary_exact'),
    col('salary_str').cast(DoubleType()).alias('salary_float'),  # potential rounding
)
df_dec.show()
# DecimalType(10,2): 10 total digits, 2 after decimal point → max 99,999,999.99

In [ ]:
# Cell 5: Schema from DataFrame — programmatic access and schema reuse

print('=== df.schema — returns a StructType object ===')
schema_obj = df_cast.schema
print(type(schema_obj))
print(schema_obj)

print('=== df.dtypes — list of (column_name, type_string) tuples ===')
print(df_cast.dtypes)

print('=== df.schema.fields — list of StructField objects ===')
for field in df_cast.schema.fields:
    print(f'  name={field.name}, dataType={field.dataType}, nullable={field.nullable}')

print('=== Access a specific field by name ===')
salary_field = df_cast.schema['salary']
print(f'salary field: name={salary_field.name}, type={salary_field.dataType}')

print('=== Schema as JSON string — useful for logging / debugging ===')
import json
schema_json = df_cast.schema.json()
print(json.dumps(json.loads(schema_json), indent=2)[:300], '...')

print('=== Reuse schema for reading new files ===')
# After verifying schema from one file, reuse for all files in same format
verified_schema = df_explicit.schema
# df_new = spark.read.schema(verified_schema).option('header', True).csv('/new/path')
print(f'Schema captured: {verified_schema.simpleString()}')

print('=== ArrayType — containsNull flag ===')
schema_arr_no_null = StructType([
    StructField('id',   IntegerType(), nullable=False),
    StructField('vals', ArrayType(IntegerType(), containsNull=False), nullable=True),
])
# containsNull=False means array ELEMENTS cannot be null (DataFrame rows still can be)
df_arr = spark.createDataFrame([(1, [1, 2, 3]), (2, [4, 5, 6])], schema_arr_no_null)
df_arr.printSchema()
df_arr.show()

print('=== size() — length of array ===')
from pyspark.sql.functions import size, array_contains
df_complex = spark.createDataFrame(
    [(1, ['a','b','c']), (2, ['x']), (3, None)],
    StructType([
        StructField('id',   IntegerType(), True),
        StructField('tags', ArrayType(StringType()), True),
    ])
)
df_complex.select(
    'id',
    size('tags').alias('num_tags'),             # -1 if null
    array_contains('tags', 'a').alias('has_a'), # null if array is null
).show()

spark.stop()
print('Done.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Integers read as strings — schema corruption after no-inferSchema read</summary>

**Situation:**
A pipeline reads a CSV daily. Without a schema, all columns default to `StringType`. A downstream `sum('revenue')` fails because you cannot sum strings. The team adds `inferSchema=True` but it is slow for the 500 MB file.

**Solution:**
```python
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType, StringType

# Define schema once (verified against sample file)
schema = StructType([
    StructField('order_id',  IntegerType(), True),
    StructField('product',   StringType(),  True),
    StructField('quantity',  IntegerType(), True),
    StructField('revenue',   DoubleType(),  True),
    StructField('order_date',DateType(),    True),
])

# Explicit schema: no sampling pass → reads 2-3x faster
df = spark.read \
    .option('header', True) \
    .option('dateFormat', 'yyyy-MM-dd') \
    .schema(schema) \
    .csv('/data/orders.csv')

total = df.agg({'revenue': 'sum'}).collect()[0][0]
print(f'Total revenue: {total:,.2f}')
```

**Exam Sub-topic:** Explicit StructType schema vs inferSchema — performance and correctness
</details>

<details>
<summary>Scenario 2: Nested JSON — accessing deep struct fields</summary>

**Situation:**
A Kafka event log is stored as Parquet. Each row has a nested `user` struct with `id`, `name`, and `address` (which itself has `city` and `country`). You need to extract city for a dashboard.

**Solution:**
```python
# Schema already embedded in Parquet — no need to specify
df = spark.read.parquet('/data/events')
df.printSchema()
# root
#  |-- event_id: string
#  |-- user: struct
#  |    |-- id: integer
#  |    |-- name: string
#  |    |-- address: struct
#  |    |    |-- city: string
#  |    |    |-- country: string

# Dot notation accesses nested struct at any depth
df.select(
    'event_id',
    'user.id',
    'user.name',
    'user.address.city',       # two levels deep
    'user.address.country'
).show()
```

**Exam Sub-topic:** Accessing nested StructType fields using dot notation string or `col()`
</details>

<details>
<summary>Scenario 3: Array column — explode product tags for frequency analysis</summary>

**Situation:**
A product catalogue has a `tags` array column. You need to count tag frequency across all products (e.g., how many products are tagged 'electronics').

**Solution:**
```python
from pyspark.sql.functions import explode, col

df = spark.read.parquet('/data/products')
# |-- product_id: integer
# |-- name: string
# |-- tags: array<string>

# explode: one row per tag
tag_counts = df \
    .select('product_id', explode('tags').alias('tag')) \
    .groupBy('tag') \
    .count() \
    .orderBy('count', ascending=False)

tag_counts.show(10)
# tag          | count
# electronics  | 1240
# portable     |  892
# ...

# explode_outer keeps null-tag rows as a single row with tag=null
df.select('product_id', explode_outer('tags').alias('tag')).show()
```

**Exam Sub-topic:** `explode()` on ArrayType; `explode_outer()` for nulls
</details>

<details>
<summary>Scenario 4: cast() silently produces NULLs — detecting bad cast results</summary>

**Situation:**
A pipeline reads employee IDs as strings. Some rows have corrupt values like `N/A` or `--`. After casting to `IntegerType`, the code proceeds without error, but downstream `groupBy('employee_id')` groups all bad values under a single `null` bucket.

**Solution:**
```python
from pyspark.sql.functions import col, count, when

df_raw = spark.read.option('header', True).csv('/data/employees.csv')
df_cast = df_raw.withColumn('emp_id', col('emp_id_str').cast('int'))

# Detect nulls introduced by failed cast (were not null before)
bad_count = df_cast \
    .filter(col('emp_id_str').isNotNull() & col('emp_id').isNull()) \
    .count()
print(f'Rows with bad emp_id values: {bad_count}')

# Show the corrupt raw values
df_cast \
    .filter(col('emp_id_str').isNotNull() & col('emp_id').isNull()) \
    .select('emp_id_str') \
    .distinct() \
    .show()
```

**Exam Sub-topic:** `cast()` returns `null` (not error) on invalid conversion; detecting cast failures
</details>

<details>
<summary>Scenario 5: MapType column — parsing dynamic key-value pairs from JSON</summary>

**Situation:**
A web analytics pipeline stores click-event properties as a map column (e.g., `{"page": "home", "button": "signup"}`). You need to extract the `page` value for each event.

**Solution:**
```python
from pyspark.sql.types import MapType, StringType, StructType, StructField
from pyspark.sql.functions import col, from_json, explode

# Option A: Map column already parsed (e.g., from Parquet)
df.select(
    'event_id',
    col('properties')['page'].alias('page'),           # key access
    col('properties')['button'].alias('button'),
).show()

# Option B: Parse JSON string column into MapType
map_schema = MapType(StringType(), StringType())
df2 = df.withColumn('props_map', from_json(col('properties_json'), map_schema))
df2.select('event_id', col('props_map')['page'].alias('page')).show()

# Option C: explode map into (key, value) rows for analysis
df.select('event_id', explode('properties').alias('prop_key', 'prop_val')).show()
# event_id | prop_key | prop_val
# 1001     | page     | home
# 1001     | button   | signup
# 1002     | page     | checkout
# ...
```

**Exam Sub-topic:** Accessing MapType values by key; `explode()` on MapType; `from_json()`
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Container for a DataFrame schema | `StructType` |
| Container for one column's metadata | `StructField(name, type, nullable)` |
| Default for `nullable` in StructField | `True` |
| How to view schema | `df.printSchema()` or `df.schema` or `df.dtypes` |
| Default type when reading CSV with no schema | `StringType` (all columns) |
| How to avoid `inferSchema` slowness | Provide explicit `StructType` schema |
| What happens when `cast()` fails? | Returns `null` — **not** an exception |
| Cast string `'NONE'` to `IntegerType` | Returns `null` |
| Access nested struct field `city` inside `address` | `df.select('address.city')` or `col('address.city')` |
| Access first element of array column `tags` | `col('tags')[0]` |
| Access map value for key `'math'` | `col('scores')['math']` |
| Expand array to one row per element | `explode('column')` |
| Keep null rows when exploding | `explode_outer('column')` |
| Expand map to one row per key-value | `explode('map_col').alias('k', 'v')` |
| Exact decimal arithmetic | `DecimalType(precision, scale)` |
| 64-bit integer type name | `LongType()` |
| Default numeric type for Python float | `DoubleType()` |
| `containsNull` in ArrayType | Whether array **elements** can be null |
| `valueContainsNull` in MapType | Whether map **values** can be null |

---

## Practice Q&A

<details>
<summary>Q1: What does printSchema() show and what do the nullable flags mean?</summary>

**A:** `printSchema()` prints a tree view of the schema with column names, data types, and `nullable` flags. Nested types are indented.

- `nullable = true` — the column can contain `null` values
- `nullable = false` — Spark expects no nulls (but does NOT enforce this at runtime unless explicitly validated; it is metadata only)
- `containsNull = true` — for ArrayType: individual array elements may be null
- `valueContainsNull = true` — for MapType: map values may be null
</details>

<details>
<summary>Q2: You read a CSV with inferSchema=True. The hire_date column comes back as StringType instead of DateType. Why and how do you fix it?</summary>

**A:** Spark's CSV inferSchema does not auto-detect date columns — it reads them as strings unless told the format. Fix:

```python
# Option 1: Explicit schema with DateType
from pyspark.sql.types import StructType, StructField, DateType, StringType
schema = StructType([StructField('hire_date', DateType(), True), ...])
df = spark.read.schema(schema).option('dateFormat', 'yyyy-MM-dd').csv(path)

# Option 2: Read as string, then convert
from pyspark.sql.functions import to_date
df = df.withColumn('hire_date', to_date(col('hire_date'), 'yyyy-MM-dd'))
```
</details>

<details>
<summary>Q3: You cast a string column to IntegerType. Some values are 'N/A'. What happens?</summary>

**A:** Spark does NOT throw an exception. Invalid cast values are silently converted to `null`. No error or warning is raised. You must explicitly check for unexpected nulls after the cast if data quality matters:

```python
df_cast = df.withColumn('age', col('age_str').cast('int'))
bad_rows = df_cast.filter(col('age_str').isNotNull() & col('age').isNull())
bad_rows.show()   # reveals rows where cast failed
```
</details>

<details>
<summary>Q4: How do you access a field two levels deep in a nested struct (e.g., user.address.city)?</summary>

**A:** Use dot notation in a string column reference or in `col()`:

```python
# Both are equivalent:
df.select('user.address.city')
df.select(col('user.address.city'))
# NOT this (does not traverse struct):
# df['user']['address']['city']   ← this works too but is less common in exam
```

The result is a flat column named `city`.
</details>

<details>
<summary>Q5: What is the difference between explode() and explode_outer() on an array column?</summary>

**A:**
- **`explode('tags')`** — produces one row per array element. If `tags` is `null` or empty, the entire row is **dropped** from output.
- **`explode_outer('tags')`** — same as `explode` but rows with `null` or empty arrays are **kept** with `null` for the exploded column.

```python
data = [(1, ['a','b']), (2, []), (3, None)]
df = spark.createDataFrame(data, ['id', 'tags'])

df.select('id', explode('tags').alias('tag')).show()
# id=1: a, b  |  id=2: (dropped)  |  id=3: (dropped)

df.select('id', explode_outer('tags').alias('tag')).show()
# id=1: a, b  |  id=2: null  |  id=3: null
```
</details>